# Dependancies
This section installs and imports all libraries needed throughout the project (data handling, visualization, ML models, deep learning, and resampling).

In [ ]:
# Install all required Python packages in one shell command via Jupyter's '!' magic.
# - kagglehub: convenient downloader/loader for Kaggle datasets
# - numpy/pandas: numerical arrays and tabular data
# - matplotlib/seaborn: plotting libraries used for visualizations
# - scikit-learn: classical ML models, preprocessing, evaluation metrics
# - imbalanced-learn: provides SMOTE for handling class imbalance
!pip install kagglehub numpy pandas matplotlib seaborn scikit-learn imbalanced-learn

In [ ]:
import os                                                # Filesystem utilities (paths, makedirs)
import numpy as np                                       # Numerical arrays and math
import pandas as pd                                      # DataFrame/Series for tabular data
import matplotlib                                        # Base matplotlib (rc settings, backends)
import matplotlib.pyplot as plt                          # Plot creation API
import seaborn as sns                                    # Higher-level statistical plotting (heatmaps etc.)
import torch                                             # PyTorch tensor library + autograd
import torch.nn as nn                                    # Neural network layers/loss building blocks
import kagglehub                                         # API to fetch Kaggle datasets
from kagglehub import KaggleDatasetAdapter               # Adapter enum to load datasets directly into pandas
from sklearn.preprocessing import StandardScaler, LabelEncoder   # Feature scaling and categorical encoding
from sklearn.model_selection import train_test_split     # Holdout split helper (imported, not always used)
from sklearn.linear_model import LogisticRegression      # Logistic Regression classifier
from sklearn.ensemble import RandomForestClassifier      # Random Forest classifier
from sklearn.metrics import *                            # Import all metrics (precision, recall, f1, roc_auc, etc.)
from sklearn.model_selection import KFold                # K-Fold cross-validation splitter
from sklearn.preprocessing import RobustScaler           # Alternative scaler robust to outliers
from imblearn.over_sampling import SMOTE                 # Synthetic Minority Oversampling Technique
from torch.utils.data import TensorDataset, DataLoader   # Wraps tensors and batches them for training

NUM_FOLDS = 4                                            # Number of folds used in cross-validation
RANDOM_STATE = 42                                        # Seed for reproducible randomness across libs
SAMPLE_SIZE = 300000 # Edit for final run, 300000-500000 # Number of rows to subsample for faster experimentation

# Detect a CUDA-enabled GPU; fall back to CPU if not available. Used to move tensors/models for NN training.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") #used to automatically detect GPU or default to CPU otherwise

### Metric Helpers
Reusable utility functions for computing metrics, saving figures, plotting confusion matrices/ROC curves/metric bars/feature importances, building K-fold splits, printing reports, and constructing model factories. Centralizing these keeps each dataset section concise and consistent.

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
  # Returns a dict of standard binary classification metrics in one call.
  return{
      "precision": precision_score(y_true, y_pred),     # TP / (TP + FP)
      "recall": recall_score(y_true, y_pred),           # TP / (TP + FN) — fraud detection rate
      "f1": f1_score(y_true, y_pred),                   # Harmonic mean of precision and recall
      "roc_auc": roc_auc_score(y_true, y_prob),         # Area under ROC; uses probability scores
      "mcc": matthews_corrcoef(y_true, y_pred),         # Balanced metric robust to class imbalance
  }

def save_fig(fig, name):
    # Save a matplotlib figure into the current RESULTS directory, then display and close it.
    path = os.path.join(RESULTS, name)                   # Build absolute path under RESULTS folder
    fig.savefig(path, dpi=150, bbox_inches="tight")     # Save with decent DPI and trimmed margins
    plt.show()                                            # Display inline in the notebook
    plt.close(fig)                                        # Free memory by closing the figure handle

def confusion_matrices(splits, Xfeatures, y, train_predict_fn, model_name, file):
    # Plots one confusion-matrix heatmap per CV fold side-by-side.
    fig, axes = plt.subplots(1, len(splits), figsize=(5 * len(splits), 4))  # One subplot per fold
    if len(splits) == 1:
        axes = [axes]                                     # Normalize axes to a list when only 1 fold

    for i, (train_idx, test_idx) in enumerate(splits):    # Iterate folds
        y_pred = train_predict_fn(train_idx, test_idx)    # Get predictions for this fold via callback
        y_test = y.iloc[test_idx]                         # Ground-truth labels for the fold's test set
        cm = confusion_matrix(y_test, y_pred)             # 2x2 matrix: TN, FP / FN, TP

        # Render the matrix as an annotated heatmap with class labels.
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i],
                    xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"])
        axes[i].set_title(f"Split {i + 1}")
        axes[i].set_xlabel("Predicted")
        axes[i].set_ylabel("Actual")

    fig.suptitle(f"{model_name} - Confusion Matrices", fontweight="bold")  # Overall title
    fig.tight_layout()                                    # Prevent overlapping labels
    save_fig(fig, file)                                   # Save and show


def plot_roc_curve(splits, Xfeatures, y, train_predict_proba_fn, model_name, file):
    # Plot an ROC curve per fold on a single axes for direct comparison.
    fig, ax = plt.subplots(figsize=(8, 6))

    for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
        y_prob = train_predict_proba_fn(train_idx, test_idx)  # Probability of fraud for test set
        y_test = y.iloc[test_idx]                              # Ground truth
        fpr, tpr, _ = roc_curve(y_test, y_prob)                # Sweep thresholds → (FPR, TPR) points
        auc_val = roc_auc_score(y_test, y_prob)                # Compute AUC for legend
        ax.plot(fpr, tpr, label=f"Split {split_num} (AUC = {auc_val:.3f})")

    ax.plot([0, 1], [0, 1], "k--", label="Random Classifier") # Diagonal reference line
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{model_name} ROC Curve")
    ax.legend(loc="lower right")
    fig.tight_layout()
    save_fig(fig, file)


def plot_metrics_bar(results_df, splits, model_name, file):
    # Grouped bar chart: one cluster per fold, one bar per metric.
    fig, ax = plt.subplots(figsize=(10, 5))

    metrics_to_plot = ["precision", "recall", "f1", "roc_auc", "mcc"]  # Metrics to show
    x = np.arange(len(splits))                            # x positions for fold groups
    bar_width = 0.15                                      # Width of each individual metric bar
    colors = ["#056517", "#de1a24", "#3b75e9", "#cad5ed", "#2832c4"]   # Distinct color per metric

    for i, metric in enumerate(metrics_to_plot):
        vals = results_df[metric].values                  # Metric values across folds
        ax.bar(x + i * bar_width, vals, bar_width, label=metric, color=colors[i])  # Offset bars

    ax.set_xlabel("Split", fontsize=12, fontweight="bold")
    ax.set_ylabel("Score", fontsize=12, fontweight="bold")
    ax.set_title(f"{model_name} Metrics", fontsize=14, fontweight="bold")
    ax.set_xticks(x + bar_width * (len(metrics_to_plot) - 1) / 2)   # Center fold tick under group
    ax.set_xticklabels([f"Split {i + 1}" for i in range(len(splits))])
    ax.set_ylim(0, 1.05)                                  # Metrics are bounded in [0,1]
    ax.legend(loc="lower right")
    fig.tight_layout()
    save_fig(fig, file)


def plot_feature_coefficients(coefs, feature_names, model_name, split_label, file, top_n=15):
    # Horizontal bar chart of top-N largest |coefficient| features for a linear model.
    fig, ax = plt.subplots(figsize=(12, 6))

    feature_names = np.array(feature_names)               # Convert to numpy for fancy indexing
    sorted_idx = np.argsort(np.abs(coefs))[::-1]          # Sort indices by |coef| descending
    top_idx = sorted_idx[:top_n]                          # Keep only the top_n indices

    top_coefs = coefs[top_idx]                            # Their actual signed coefficient values
    top_names = feature_names[top_idx]                    # Their corresponding feature names
    bar_colors = ["#de1a24" if c < 0 else "#056517" for c in top_coefs]  # Red for negative, green for positive

    ax.barh(top_names, top_coefs, color=bar_colors, edgecolor="black")
    ax.set_xlabel("Coefficient Value", fontsize=12)
    ax.set_title(f"{model_name} - Top {top_n} Coefficients ({split_label})",
                 fontsize=14, fontweight="bold")
    ax.invert_yaxis()                                     # Largest at top of chart
    fig.tight_layout()
    save_fig(fig, file)

def _make_cached_predictor(fold_predictions, key, n_folds):
    # Returns a predictor function that yields cached predictions per fold call,
    # so we don't re-train models inside the plotting helpers.
    preds = [fold_predictions[i + 1][key] for i in range(n_folds)]  # Pre-fetch fold predictions in order
    call_idx = [0]                                        # Mutable counter via list (closure)
    def predictor(train_idx, test_idx):
        result = preds[call_idx[0]]                       # Use the next cached prediction
        call_idx[0] += 1                                  # Advance pointer
        return result
    return predictor

def create_kfold_splits(X, y, n_folds=NUM_FOLDS, random_state=RANDOM_STATE):
    # Build a deterministic list of (train_idx, test_idx) pairs using shuffled K-Fold.
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    return list(kf.split(X, y))

def print_classification_report(y_true, y_pred):
    # Pretty-print sklearn's per-class precision/recall/F1/support table.
    print(classification_report(y_true, y_pred,target_names=["Legitimate", "Fraudulent"],zero_division=0))

def print_summary_table(results_dset2_df, model_name):
    # Prints mean and std per metric across all folds in a tidy aligned table.
    print(f"\n {model_name} Summary Across All Folds")
    print(f"  {'Metric':<12} {'Mean':>10} {'Std':>10}")
    print(f"  {'-' * 32}")
    for metric in ["precision", "recall", "f1", "mcc", "roc_auc"]:
        mean_val = results_dset2_df[metric].mean()
        std_val = results_dset2_df[metric].std()
        print(f"  {metric:<12} {mean_val:>10.4f} {std_val:>10.4f}")

def _build_lr():
    # Factory for a baseline Logistic Regression with fixed hyperparameters for reproducibility.
    return LogisticRegression(
        C=1.0,                                            # Inverse regularization strength
        solver="lbfgs",                                  # Default optimizer for L2-regularized LR
        max_iter=1000,                                    # Allow more iterations to converge on scaled data
        random_state=42                                   # Seed for reproducibility
    )

def _build_rf():
    # Factory for a baseline Random Forest with sensible defaults and parallel training.
    return RandomForestClassifier(
        n_estimators=100,                                 # Number of trees in the forest
        max_depth=None,                                   # Trees grow until pure or min_samples_leaf reached
        min_samples_split=2,                              # Minimum samples to split an internal node
        min_samples_leaf=1,                               # Minimum samples allowed in a leaf
        max_features="sqrt",                             # sqrt(num_features) features considered per split
        random_state=42,                                  # Seed for reproducibility
        n_jobs= -1                                        # Use all CPU cores
    )

# Feed-Forward Neural Network
Defines the FFNN architecture and training/prediction helpers that will be reused across all three datasets.

In [ ]:
HIDDEN1 = 64                                              # Neurons in first hidden layer
HIDDEN2 = 32                                              # Neurons in second hidden layer
DROPOUT = 0.3                                             # Dropout probability for regularization
EPOCHS = 50                                               # Total training epochs per fold
BATCH_SIZE = 128                                          # Mini-batch size during training
LEARNING_RATE = 1e-3                                      # Adam learning rate
WEIGHT_DECAY = 1e-4                                       # L2 weight regularization in Adam

class FraudDetectorNN(nn.Module):
    # A small feed-forward classifier producing a single sigmoid-activated probability.
    def __init__(self, input_dim, hidden1=HIDDEN1, hidden2=HIDDEN2, dropout=DROPOUT):
        super().__init__()                                # Initialize nn.Module base
        # Stack of Linear → BatchNorm → ReLU → Dropout blocks, ending with Sigmoid output.
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden1),                # Input → Hidden1
            nn.BatchNorm1d(hidden1),                      # Stabilize activations across batch
            nn.ReLU(),                                    # Non-linearity
            nn.Dropout(dropout),                          # Randomly drop activations during training

            nn.Linear(hidden1, hidden2),                  # Hidden1 → Hidden2
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden2, 1),                        # Hidden2 → 1 logit
            nn.Sigmoid(),                                 # Squash to probability in [0, 1]
        )

    def forward(self, x):
        # Forward pass; squeeze trailing dim so output shape is (batch_size,).
        return self.network(x).squeeze(-1)

def _set_seeds():
    # Seed PyTorch + NumPy (and CUDA if present) so NN training is reproducible.
    torch.manual_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_STATE)

def _train_nn(X_train, y_train, input_dim, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY):
    # Train the FFNN for one fold and return the trained model plus per-epoch loss history.
    _set_seeds()                                          # Reset seeds for deterministic training
    model = FraudDetectorNN(input_dim).to(DEVICE)         # Build model and move to GPU/CPU
    criterion = nn.BCELoss()                              # Binary cross-entropy loss
    optimizer = torch.optim.Adam(model.parameters(),lr=lr, weight_decay=weight_decay)  # Adam optimizer
    # Convert pandas/numpy inputs into raw numpy arrays for tensor construction.
    X_arr = X_train.values if hasattr(X_train, "values") else np.array(X_train)
    y_arr = y_train.values if hasattr(y_train, "values") else np.array(y_train)
    X_tensor = torch.FloatTensor(X_arr)                   # Features as float32 tensor
    y_tensor = torch.FloatTensor(y_arr)                   # Targets as float32 tensor (BCE expects floats)
    dataset = TensorDataset(X_tensor, y_tensor)           # Pair (X, y) into one dataset
    g = torch.Generator().manual_seed(RANDOM_STATE)       # Seeded RNG for shuffle reproducibility
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, generator=g)  # Mini-batch iterator
    epoch_losses = []                                     # Track average loss per epoch
    model.train()                                         # Enable dropout/batchnorm training mode

    for epoch in range(epochs):
        running_loss = 0.0                                # Sum of per-sample losses
        n_samples = 0                                     # Total samples seen this epoch

        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)                  # Move batch to compute device
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()                         # Clear previous gradients
            y_hat = model(X_batch)                        # Forward pass → predicted probabilities
            loss = criterion(y_hat, y_batch)              # Compute BCE loss
            loss.backward()                               # Backprop
            optimizer.step()                              # Update parameters
            running_loss += loss.item() * len(X_batch)    # Accumulate weighted by batch size
            n_samples += len(X_batch)

        avg_loss = running_loss / n_samples               # Mean loss across the epoch
        epoch_losses.append(avg_loss)

        # Log progress every 10 epochs and after the first epoch for early visibility.
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"      Epoch {epoch + 1:>3}/{epochs}  |  Loss: {avg_loss:.6f}")

    return model, epoch_losses

def _predict(model, X_test):
    # Run inference: returns hard predictions and probabilities for the test set.
    model.eval()                                          # Disable dropout/batchnorm updates
    with torch.no_grad():                                 # Disable autograd to save memory/time
        X_arr = X_test.values if hasattr(X_test, "values") else np.array(X_test)
        X_tensor = torch.FloatTensor(X_arr).to(DEVICE)
        y_prob = model(X_tensor).cpu().numpy()            # Move probs back to CPU as numpy
    y_pred = (y_prob >= 0.5).astype(int)                  # Threshold at 0.5 for binary class
    return y_pred, y_prob

def make_predictor(key):
  # Like _make_cached_predictor but reads from the global `split_predictions` dict (used in Dataset 3).
  predictions = [split_predictions[i+1][key] for i in range(len(splits))]
  call_idx = [0]

  def predictor(train_idx, test_idx):
    result = predictions[call_idx[0]]
    call_idx[0] += 1
    return result
  return predictor

## Load All Data
Pull all three Kaggle datasets directly into pandas DataFrames using `kagglehub`.

In [ ]:
# Dataset 1 - Online payments fraud (synthetic PaySim style log).
Dataset1 =kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "rupakroy/online-payments-fraud-detection-dataset", "PS_20174392719_1491204439457_log.csv",)

In [ ]:
# Dataset 2 - Credit card fraud detection 2023 (already class-balanced).
Dataset2 = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "nelgiriyewithana/credit-card-fraud-detection-dataset-2023","creditcard_2023.csv",)

In [ ]:
# Dataset 3 - Classic European credit card fraud dataset (PCA features V1..V28 + Time + Amount).
Dataset3 = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "whenamancodes/fraud-detection", "creditcard.csv",)

# Dataset 1
Online payments dataset: contains transaction `type`, `amount`, balances, and a binary `isFraud` target.

In [ ]:
# Subsample to SAMPLE_SIZE rows for tractable runtimes; clip to dataset length if smaller.
df1 = Dataset1.sample(n=min(SAMPLE_SIZE, len(Dataset1)), random_state=42)

df1.head()                                                # Quick visual sanity check of first rows

## Preprocessing

In [ ]:
RESULTS = "Dataset1 results"                             # Folder name where Dataset 1 figures/CSVs are saved
os.makedirs(RESULTS, exist_ok=True)                       # Create directory if missing

#Cleaning dataset
missing = df1.isnull().sum()                              # Count missing values per column
total_missing = missing.sum()                             # Total missing across the frame
print(f"\nMissing values: {total_missing}")
if total_missing > 0:
    print(missing[missing > 0])                           # Show only columns that actually have NaNs

# Check for duplicate rows
duplicates = df1.duplicated().sum()                       # Count exact duplicate rows
print(f"Duplicate rows: {duplicates:,}")
if duplicates > 0:
    print("  → Dropping duplicates")
    df = df.drop_duplicates().reset_index(drop=True)      # NOTE: variable name 'df' here is a leftover bug; runs only if duplicates exist
    print(f"  → New shape: {df.shape[0]:,} rows")

# There are no missing rows or duplicates, therefore null/missing/duplicate cleaning not needed

In [ ]:
# Drop high-cardinality identifiers (account names) and the simulation step counter that aren't predictive.
df1 = df1.drop(columns=['nameOrig', 'nameDest', 'step'])

df1.head()                                                # Inspect after dropping columns

# Encode the categorical 'type' column as integers because sklearn models need numeric inputs.
if 'type' in df1.columns:
    print(f"Original 'type' column values: {df1['type'].unique()}")          # Distinct categories present
    print(f"Value counts:\n{df1['type'].value_counts()}")                    # Frequency per category

    label_encoder = LabelEncoder()                                            # sklearn integer encoder
    df1['type_encoded'] = label_encoder.fit_transform(df1['type'])            # Add encoded column
    print(f"\nLabel encoding applied:")
    for i, category in enumerate(label_encoder.classes_):                     # Print mapping for transparency
        print(f"  {category} → {i}")

    df1 = df1.drop(columns=['type'])                                          # Remove original string column
    print(f"\nDropped original 'type' column")
else:
    print("No 'type' column found for encoding")

df1.head()

### Data Analysis
Quick look at how imbalanced the target is. Fraud is extremely rare in this dataset.

In [ ]:
print(f"Total transactions: {len(Dataset1):,}")          # Total row count in the original (not subsample)
class_counts = Dataset1["isFraud"].value_counts().sort_index()       # 0/1 counts
class_pct = Dataset1["isFraud"].value_counts(normalize=True).sort_index() * 100  # 0/1 percentages
print(f"Legitimate transactions: {class_counts[0]:,} ({class_pct[0]:.4f}%)")
print(f"Fraudulent transactions: {class_counts[1]:,} ({class_pct[1]:.4f}%)")

### Feature Engineering
Build domain-informed features that highlight fraud-typical patterns: balance changes, ratios, log transforms, mismatches, and interactions. These engineered features tend to give linear models more separability.

In [ ]:
#Feture Engineering
df1 = df1.assign(
    balance_change_orig=df1['newbalanceOrig'] - df1['oldbalanceOrg']    # Net change on the originator account
)

df1 = df1.assign(
    balance_change_dest=df1['newbalanceDest'] - df1['oldbalanceDest']   # Net change on the destination account
)

df1 = df1.assign(
    amount_to_balance_ratio=df1['amount'] / (df1['oldbalanceOrg'] + 1)  # How big is the txn vs originator's balance? (+1 avoids /0)
)

df1 = df1.assign(
    amount_to_dest_ratio=df1['amount'] / (df1['oldbalanceDest'] + 1)    # How big is the txn vs destination's balance?
)

df1 = df1.assign(
    abs_balance_change_orig=np.abs(df1['balance_change_orig']),         # Magnitude of orig balance change
    abs_balance_change_dest=np.abs(df1.get('balance_change_dest', 0))   # Magnitude of dest balance change
)

amount_mean = df1['amount'].mean()                                       # Mean transaction size
amount_std = df1['amount'].std()                                         # Std of transaction sizes

df1 = df1.assign(
    is_large_transaction=(df1['amount'] > amount_mean + 2 * amount_std).astype(int)  # Flag amount > μ+2σ
)

df1 = df1.assign(
    balance_mismatch_orig=(
        np.abs(df1['newbalanceOrig'] - (df1['oldbalanceOrg'] - df1['amount'])) > 0.01  # Detect impossible bookkeeping
    ).astype(int)
)

df1 = df1.assign(
    is_zero_balance_after=(df1['newbalanceOrig'] == 0).astype(int)       # Account drained to zero after txn
)

df1 = df1.assign(
    type_amount_interaction=df1['type_encoded'] * np.log1p(df1['amount'])  # Interaction between txn type and log(amount)
)

df1 = df1.assign(
    amount_change_interaction=df1['amount'] * df1['balance_change_orig']   # Joint effect of size and balance shift
)

df1 = df1.assign(
    log_amount=np.log1p(df1['amount']),                                    # log(1+x) compresses skewed amount
    log_oldbalance_orig=np.log1p(df1['oldbalanceOrg']),
    log_oldbalance_dest=np.log1p(df1.get('oldbalanceDest', 0))
)

avg_transaction_amount = df1['amount'].mean()                              # Reference average for ratio feature
df1 = df1.assign(
    amount_to_avg_ratio=df1['amount'] / (avg_transaction_amount + 1)       # Normalized amount
)

df1.head()                                                                 # Show engineered feature columns

## Data Split
Separate features from target, build K-Fold splits, and define a per-fold scaling+SMOTE pipeline that prevents leakage.

In [ ]:
# Features and Targets
X = df1.drop(columns=['isFraud'])                         # All columns except the label become features
Y = df1['isFraud']                                        # Binary fraud label

Y = Y.astype(int)                                         # Ensure integer dtype

feature_names = X.columns.tolist()                        # Cache feature names for plotting

print(f"Features shape: {X.shape}")
print(f"Target shape: {Y.shape}")
print(f"\nTarget distribution:")
print(f"  Non-Fraud (0): {(Y == 0).sum():,} ({(Y == 0).mean() * 100:.4f}%)")
print(f"  Fraud (1):     {(Y == 1).sum():,} ({(Y == 1).mean() * 100:.4f}%)")

def scale_resample(X, y, train_idx, test_idx):
    # Per-fold preprocessing: fit StandardScaler on train only, then SMOTE the training set.
    # This prevents test-set information from leaking into the scaler or oversampler.
    feature_names = X.columns.tolist()
    X_train_raw = X.iloc[train_idx]                       # Train features for this fold
    X_test_raw = X.iloc[test_idx]                         # Test features for this fold
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    scaler = StandardScaler()                             # Zero-mean / unit-variance scaler
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_raw))   # Fit + transform train
    X_test_scaled = pd.DataFrame(scaler.transform(X_test_raw))         # Transform test using train stats only

    smote = SMOTE(random_state=42)                        # Oversample minority class via synthetic samples
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

    return X_train_resampled, y_train_resampled, X_test_scaled, y_test

splits = create_kfold_splits(X, Y)                        # Build (train_idx, test_idx) pairs

for i, (train_idx, test_idx) in enumerate(splits, start=1):
    train_fraud = Y.iloc[train_idx].sum()                 # Count fraud cases in train fold
    test_fraud = Y.iloc[test_idx].sum()                   # Count fraud cases in test fold
    test_pct = test_fraud / len(test_idx) * 100           # Fraud rate in the test fold

def plot_feature_importances(importances, feature_names, model_name, fold_label, filename, top_n=15):
  # Horizontal bar chart of Random Forest's Gini importances (top N features).
  fig, ax = plt.subplots(figsize=(12, 6))
  sorted_idx = np.argsort(importances)[::-1][:top_n]      # Indices of top_n importances
  ax.barh(range(top_n), importances[sorted_idx], color="#3498db", edgecolor="black", linewidth=0.5)
  ax.set_yticks(range(top_n))
  ax.set_yticklabels([feature_names[i] for i in sorted_idx])
  ax.set_xlabel("Gini Importance", fontsize=12)
  ax.set_title(f"{model_name} - Top {top_n} Feature Importances ({fold_label})", fontsize=14, fontweight="bold")
  ax.invert_yaxis()                                       # Most important on top
  fig.tight_layout()
  save_fig(fig, filename)

## Models

### Logistic Regreession

In [ ]:
lr_results_ds1 = []                                       # Per-fold metric dicts will accumulate here
lr_last_model = None                                      # Keep handle to most recently trained model for coefs

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    # Get preprocessed data for this fold
    X_train_rs, y_train_rs, X_test_scaled, y_test = scale_resample(X, Y, train_idx, test_idx)

    # Train the model
    model = _build_lr()                                   # Fresh LR instance with fixed hyperparams
    model.fit(X_train_rs, y_train_rs)                     # Fit on SMOTE-balanced training data
    lr_last_model = model  # Keep the last trained model

    # Predict and evaluate
    y_pred = model.predict(X_test_scaled)                 # Hard 0/1 predictions on real test set
    y_prob = model.predict_proba(X_test_scaled)[:, 1]     # Probability of fraud class
    metrics = compute_metrics(y_test, y_pred, y_prob)     # Compute all evaluation metrics
    metrics["split"] = split_num                          # Tag with the fold number
    lr_results_ds1.append(metrics)

# Convert results to a DataFrame for easy analysis
lr_results_ds1_df = pd.DataFrame(lr_results_ds1)
lr_results_ds1_df.to_csv(os.path.join(RESULTS, "lr_results_ds1.csv"), index=False)  # Persist for reporting
#print_summary_table(lr_results_ds1_df, "Logistic Regression")

In [ ]:
def _lr_predict(train_idx, test_idx):
    """Predict using Logistic Regression for a given fold"""
    # Re-runs the full per-fold pipeline so plotting helpers have a fresh prediction callback.
    X_train_rs, y_train_rs, X_test_scaled, y_test = scale_resample(X, Y, train_idx, test_idx)
    model = _build_lr()
    model.fit(X_train_rs, y_train_rs)
    return model.predict(X_test_scaled)

def _lr_predict_proba(train_idx, test_idx):
    """Predict probabilities using Logistic Regression for a given fold"""
    X_train_rs, y_train_rs, X_test_scaled, y_test = scale_resample(X, Y, train_idx, test_idx)
    model = _build_lr()
    model.fit(X_train_rs, y_train_rs)
    return model.predict_proba(X_test_scaled)[:, 1]

# Generate the four standard plots: confusion matrices, ROC, metric bars, top coefficients.
confusion_matrices(splits, X, Y, _lr_predict,"Logistic Regression", "d1_lr_confusion_matrices.png")
plot_roc_curve(splits, X, Y, _lr_predict_proba,"Logistic Regression", "d1_lr_roc_curves.png")
plot_metrics_bar(lr_results_ds1_df, splits, "Logistic Regression","d1_lr_metrics_by_fold.png")
plot_feature_coefficients(lr_last_model.coef_[0], feature_names,"Logistic Regression", f"Fold {len(splits)}","14_lr_feature_coefficients.png")

### Random Forest Classifier

In [ ]:
#training random forest model
rf_results = []                                           # Per-fold metric records
rf_last_model = None                                      # Holder for the latest trained RF (for importances)
rf_fold_predictions = {}                                  # Cache predictions to avoid retraining for plots

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    X_train, y_train, X_test, y_test = scale_resample(X, Y, train_idx, test_idx)
    model = _build_rf()                                   # Fresh forest per fold
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["split"] = split_num
    rf_results.append(metrics)
    rf_fold_predictions[split_num] = {"y_pred": y_pred, "y_prob": y_prob}  # Cache for plotting
    rf_last_model = model

rf_results_ds1_df = pd.DataFrame(rf_results)
rf_results_ds1_df.to_csv(os.path.join(RESULTS, "rf_results.csv"), index=False)
print_summary_table(rf_results_ds1_df, "Random Forest")

In [ ]:
# Same four plots as LR, using cached fold predictions to avoid re-training the forest.
confusion_matrices(splits, X, Y,_make_cached_predictor(rf_fold_predictions, "y_pred", len(splits)),"Random Forest", "d1_rf_confusion_matrices.png")
plot_roc_curve(splits, X, Y,_make_cached_predictor(rf_fold_predictions, "y_prob", len(splits)),"Random Forest", "d1_rf_roc_curves.png")
plot_metrics_bar(rf_results_ds1_df, splits, "Random Forest", "d1_rf_metrics_by_fold.png")
plot_feature_importances(rf_last_model.feature_importances_, feature_names,"Random Forest", f"Fold {len(splits)}","d1_rf_feature_importances.png")

### Feed-Forward Neural Network

In [ ]:
input_dim = X.shape[1]                                    # Number of features = NN input size
nn_results = []                                           # Per-fold metric records
nn_fold_predictions = {}                                  # Cached predictions per fold
all_epoch_losses = {}                                     # Loss curves per fold for plotting

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    X_train, y_train, X_test, y_test = scale_resample(X, Y, train_idx, test_idx)
    model, epoch_losses = _train_nn(X_train, y_train, input_dim)   # Train one FFNN for this fold
    y_pred, y_prob = _predict(model, X_test)              # Inference on the held-out fold
    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["split"] = split_num
    nn_results.append(metrics)
    nn_fold_predictions[split_num] = {"y_pred": y_pred, "y_prob": y_prob}
    all_epoch_losses[split_num] = epoch_losses

nn_results_ds1_df = pd.DataFrame(nn_results)
nn_results_ds1_df.to_csv(os.path.join(RESULTS, "nn_results.csv"), index=False)
print_summary_table(nn_results_ds1_df, "Feed Forward Neural Network")

In [ ]:
# Plot the training loss for each fold on a single axes for comparison.
fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.get_cmap("tab10")                              # Categorical colormap
for split_num in sorted(all_epoch_losses.keys()):
    losses = all_epoch_losses[split_num]
    ax.plot(range(1, len(losses) + 1), losses,
            label=f"Fold {split_num}", color=cmap((split_num - 1) % 10), linewidth=1.3)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Binary Cross-Entropy Loss", fontsize=12)
ax.set_title("Feed-Forward Neural Network - Training Loss Curves",
             fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=8); ax.grid(True, alpha=0.3)
fig.tight_layout()
save_fig(fig, "d1_nn_training_loss.png")

# Diagnostic plots for the NN (CMs, ROC, metrics bar) using cached predictions.
confusion_matrices(splits, X, Y,_make_cached_predictor(nn_fold_predictions, "y_pred", len(splits)),"Feed-Forward Neural Network", "d1_nn_confusion_matrices.png")
plot_roc_curve(splits, X, Y,_make_cached_predictor(nn_fold_predictions, "y_prob", len(splits)),"Feed-Forward Neural Network", "d1_nn_roc_curves.png")
plot_metrics_bar(nn_results_ds1_df, splits, "Feed-Forward Neural Network","d1_nn_metrics_by_fold.png")
print("\nFeed-Forward Neural Network complete.")

In [ ]:
# Aggregate the three model result frames into a single mean-metric summary table for Dataset 1.
summary_frames = {"Logistic Regression": lr_results_ds1_df,"Random Forest": rf_results_ds1_df,"Feed-Forward NN": nn_results_ds1_df}
metric_cols = ["precision", "recall", "f1", "mcc", "roc_auc"]
summary = pd.DataFrame({
    name: frame[metric_cols].mean()                       # Mean of each metric across folds
    for name, frame in summary_frames.items()
}).T                                                       # Transpose so models are rows

print("Mean metrics across all 4 folds:")
print(summary.round(4))
summary.to_csv(os.path.join(RESULTS, "summary_mean_metrics.csv"))

#Dataset 2
Already-balanced credit card fraud dataset (synthetic 50/50). No SMOTE needed; just scale and split.

##Preprocessing

In [ ]:
# Subsample dataset 2 for faster experimentation (or use full Dataset2 for the final run).
dset2_df = Dataset2.sample(n=min(SAMPLE_SIZE, len(Dataset2)), random_state=42) #Edit for final run, dset2_df = Dataset2

dset2_df.head()

In [ ]:
RESULTS = "Dataset2 results"                              # Switch RESULTS folder so subsequent save_fig calls land in dataset 2
os.makedirs(RESULTS, exist_ok=True)

###Dataset Cleaning

In [ ]:
# drop ID column to prevent memorization
if "id" in dset2_df.columns:
    dset2_df = dset2_df.drop(columns=["id"])              # Row IDs would let the model 'memorize' rows

# check for missing values
missing = dset2_df.isnull().sum()
total_missing = missing.sum()
print(f"\nMissing values: {total_missing}")
if total_missing > 0:
    print(missing[missing > 0])

# check for duplicate rows to prevent information leakage
duplicates = dset2_df.duplicated().sum()
print(f"Duplicate rows: {duplicates:,}")
if duplicates > 0:
    dset2_df = dset2_df.drop_duplicates().reset_index(drop=True)   # Remove duplicate transactions
    print(f"  -> New shape: {dset2_df.shape[0]:,} rows")

###Dataset Analysis

In [ ]:
# Visualize how legitimate vs fraudulent classes are distributed (this dataset is balanced ~50/50).
class_counts = dset2_df["Class"].value_counts().sort_index()
class_pct = dset2_df["Class"].value_counts(normalize=True).sort_index() * 100
colors = ["#7ed957", "#d236b4"]
labels = ["Legitimate (0)", "Fraudulent (1)"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(labels, class_counts.values, color=colors, edgecolor="black")  # Bar chart of raw counts
axes[0].set_title("Transaction Class Counts", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Count")
for i, (cnt, pct) in enumerate(zip(class_counts.values, class_pct.values)):
    # Annotate each bar with raw count and percentage.
    axes[0].text(i, cnt + cnt * 0.01, f"{cnt:,}\n({pct:.3f}%)", ha="center", fontsize=10)

axes[1].pie(class_counts.values, labels=labels, colors=colors, autopct="%1.3f%%", startangle=90)  # Pie of proportion
axes[1].set_title("Class Proportion", fontsize=14, fontweight="bold")

fig.suptitle("Pre-Balanced Class Distribution in Dataset 2 (~50/50)",fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "Dataset2_class_distribution.png")

In [ ]:
# Compare distribution of `Amount` between legitimate and fraud at full range and zoomed-in to <= $500.
legit = dset2_df[dset2_df["Class"] == 0]["Amount"]        # Amounts for legit transactions
fraud = dset2_df[dset2_df["Class"] == 1]["Amount"]        # Amounts for fraudulent transactions

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].hist(legit, bins=80, alpha=0.6, color="#7ed957", label="Legitimate", density=True)
axes[0].hist(fraud, bins=80, alpha=0.6, color="#db2a9d", label="Fraudulent", density=True)
axes[0].set_title("Amount Distribution (Full Range)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Amount"); axes[0].set_ylabel("Density"); axes[0].legend()

#given that the majority of transactions seem to appear below $500 this range is used for cleaerer visualization
axes[1].hist(legit[legit <= 500], bins=80, alpha=0.6, color="#7ed957", label="Legitimate", density=True)
axes[1].hist(fraud[fraud <= 500], bins=80, alpha=0.6, color="#d236b4", label="Fraudulent", density=True)
axes[1].set_title("Amount Distribution (≤ $500)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Amount"); axes[1].set_ylabel("Density"); axes[1].legend()

fig.suptitle("Transaction Amount: Legitimate vs Fraudulent", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "Dataset2_amount_distribution.png")

In [ ]:
# Identify which V1..V28 + Amount features correlate most with the fraud label.
v_cols = [f"V{i}" for i in range(1, 29)]                  # V1..V28 PCA-style anonymized features
feature_cols = v_cols + ["Amount"]
corr_with_class = (
    dset2_df[feature_cols + ["Class"]].corr()["Class"].drop("Class").sort_values()
)

print("  Features most negatively correlated with Class(fraud):")
for feat, corr in corr_with_class.head(5).items():
    print(f"    {feat:<8}  r = {corr:+.4f}")

print("\n  Features most positively correlated with Class(fraud):")
for feat, corr in corr_with_class.tail(5).items():
    print(f"    {feat:<8}  r = {corr:+.4f}")

#the green bar increases a prediction of fraud while the red increses the prediction of a legitimate transaction
fig, ax = plt.subplots(figsize=(12, 6))
bar_colors = ["#d236b4" if v < 0 else "#7ed957" for v in corr_with_class.values]
ax.barh(corr_with_class.index, corr_with_class.values, color=bar_colors, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Pearson Correlation with Class", fontsize=12)
ax.set_title("Feature Correlation with Fraud Label", fontsize=14, fontweight="bold")
ax.axvline(x=0, color="black", linewidth=0.8)             # Reference line at zero correlation
fig.tight_layout()
save_fig(fig, "Dataset2_feature_correlation_with_class.png")

###Preparing Data for Training Models

In [ ]:
X = dset2_df.drop(columns=["Class"])                      # Feature matrix
y = dset2_df["Class"]                                     # Binary fraud label

In [ ]:
#dataset already balanced hence no SMOTE
def scale_split(X, y, train_idx, test_idx):
    # Per-fold scaling-only pipeline (no resampling) since classes are already balanced.
    feature_names = X.columns.tolist()
    X_train_raw = X.iloc[train_idx]
    X_test_raw = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train_raw),                # Fit + transform train fold
        columns=feature_names, index=X_train_raw.index,
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test_raw),                     # Apply same scaler to test fold
        columns=feature_names, index=X_test_raw.index,
    )

    return X_train_scaled, y_train, X_test_scaled, y_test, scaler

splits = create_kfold_splits(X, y)                        # Build new K-Fold splits for Dataset 2
for i, (train_idx, test_idx) in enumerate(splits, start=1):
    train_fraud = y.iloc[train_idx].sum()
    test_fraud = y.iloc[test_idx].sum()
    test_pct = test_fraud / len(test_idx) * 100

###Evaluation Helpers
Dataset 2 redefines slightly different versions of the plotting helpers (different palette/labels). They function the same way as the originals.

In [ ]:
def plot_confusion_matrices(splits, X_features, y, train_and_predict_fn, model_name, filename):
    # Confusion-matrix grid (one heatmap per fold) with no colorbar to keep the figure compact.
    fig, axes = plt.subplots(1, len(splits), figsize=(4 * len(splits), 4))
    if len(splits) == 1:
        axes = [axes]

    for i, (train_idx, test_idx) in enumerate(splits, start=1):
        y_pred = train_and_predict_fn(train_idx, test_idx)
        y_test = y.iloc[test_idx]
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i - 1], xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"], cbar=False)
        axes[i - 1].set_title(f"Fold {i}", fontsize=11, fontweight="bold")
        axes[i - 1].set_ylabel("Actual"); axes[i - 1].set_xlabel("Predicted")

    fig.suptitle(f"{model_name} - Confusion Matrices", fontsize=16, fontweight="bold", y=1.02)
    fig.tight_layout()
    save_fig(fig, filename)

def plot_roc_curves(splits, X_features, y, train_and_predict_proba_fn, model_name, filename):
    # Plot ROC curves for each fold along with a random-classifier diagonal reference.
    fig, ax = plt.subplots(figsize=(8, 6))

    for i, (train_idx, test_idx) in enumerate(splits, start=1):
        y_prob = train_and_predict_proba_fn(train_idx, test_idx)
        y_test = y.iloc[test_idx]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc_val = roc_auc_score(y_test, y_prob)
        ax.plot(fpr, tpr, label=f"Fold {i} (AUC={auc_val:.4f})", alpha=0.7)

    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random Classifier")
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title(f"{model_name} ROC Curves", fontsize=14, fontweight="bold")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    save_fig(fig, filename)

def plot_metrics_bars(results_dset2_df, splits, model_name, filename):
    # Grouped metric bar chart used for Dataset 2 (alternative palette).
    fig, ax = plt.subplots(figsize=(14, 5))
    metrics_to_plot = ["precision", "recall", "f1", "mcc", "roc_auc"]
    x = np.arange(len(splits))
    width = 0.15
    colors_metrics = ["#3498db", "#7ed957", "#d236b4", "#9b59b6", "#f39c12"]

    for i, metric in enumerate(metrics_to_plot):
        ax.bar(x + i * width, results_dset2_df[metric].values, width,
               label=metric.upper().replace("_", "-"), color=colors_metrics[i])

    ax.set_xlabel("Fold", fontsize=12); ax.set_ylabel("Score", fontsize=12)
    ax.set_title(f"{model_name} - Metrics by Fold", fontsize=14, fontweight="bold")
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels([f"F{i+1}" for i in range(len(splits))])
    ax.legend(loc="lower right"); ax.set_ylim(0, 1.05)
    fig.tight_layout()
    save_fig(fig, filename)

def plot_feature_coefficients(coefs, feature_names, model_name, fold_label, filename, top_n=15):
    # NOTE: Re-defines the earlier plot_feature_coefficients with Dataset-2 colors. The latest
    # definition wins for any subsequent calls in the notebook.
    fig, ax = plt.subplots(figsize=(12, 6))
    sorted_idx = np.argsort(np.abs(coefs))[::-1]
    ax.barh(range(top_n), coefs[sorted_idx[:top_n]], color=["#d236b4" if c < 0 else "#7ed957"
                   for c in coefs[sorted_idx[:top_n]]],edgecolor="black", linewidth=0.5)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([feature_names[i] for i in sorted_idx[:top_n]])
    ax.set_xlabel("Coefficient Value", fontsize=12)
    ax.set_title(f"{model_name} - Top {top_n} Feature Coefficients ({fold_label})", fontsize=14, fontweight="bold")
    ax.invert_yaxis()
    fig.tight_layout()
    save_fig(fig, filename)

##Logistic Regression

In [ ]:
feature_names = X.columns.tolist()                        # Refresh feature names for Dataset 2 X
lr_results = []
lr_last_model = None

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    X_train, y_train, X_test, y_test, _ = scale_split(X, y, train_idx, test_idx)  # Scale-only split
    model = _build_lr()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["split"] = split_num
    lr_results.append(metrics)
    lr_last_model = model

lr_dset2_df = pd.DataFrame(lr_results)
lr_dset2_df.to_csv(os.path.join(RESULTS, "lr_results.csv"), index=False)
print_summary_table(lr_dset2_df, "Logistic Regression")

In [ ]:
# Re-trained predict callbacks for plotting helpers (avoid recomputing predictions caches here).
def _lr_predict(train_idx, test_idx):
    X_tr, y_tr, X_te, _, _ = scale_split(X, y, train_idx, test_idx)
    m = _build_lr(); m.fit(X_tr, y_tr)
    return m.predict(X_te)

def _lr_predict_proba(train_idx, test_idx):
    X_tr, y_tr, X_te, _, _ = scale_split(X, y, train_idx, test_idx)
    m = _build_lr(); m.fit(X_tr, y_tr)
    return m.predict_proba(X_te)[:, 1]

plot_confusion_matrices(splits, X, y, _lr_predict,"Logistic Regression", "11_lr_confusion_matrices.png")
plot_roc_curves(splits, X, y, _lr_predict_proba,"Logistic Regression", "12_lr_roc_curves.png")
plot_metrics_bars(lr_dset2_df, splits, "Logistic Regression","13_lr_metrics_by_fold.png")
plot_feature_coefficients(lr_last_model.coef_[0], feature_names,"Logistic Regression", f"Fold {len(splits)}","14_lr_feature_coefficients.png")

##Random forest

In [ ]:
rf_results = []
rf_last_model = None
rf_fold_predictions = {}

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    X_train, y_train, X_test, y_test, _ = scale_split(X, y, train_idx, test_idx)
    model = _build_rf()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["split"] = split_num
    rf_results.append(metrics)
    rf_fold_predictions[split_num] = {"y_pred": y_pred, "y_prob": y_prob}
    rf_last_model = model

rf_dset2_df = pd.DataFrame(rf_results)
rf_dset2_df.to_csv(os.path.join(RESULTS, "rf_results.csv"), index=False)
print_summary_table(rf_dset2_df, "Random Forest")

In [ ]:
# RF diagnostic plots reusing the earlier (Dataset 1) helpers + cached predictions.
confusion_matrices(splits, X, y,_make_cached_predictor(rf_fold_predictions, "y_pred", len(splits)),"Random Forest", "15_rf_confusion_matrices.png")
plot_roc_curve(splits, X, y,_make_cached_predictor(rf_fold_predictions, "y_prob", len(splits)),"Random Forest", "16_rf_roc_curves.png")
plot_metrics_bar(rf_dset2_df, splits, "Random Forest", "17_rf_metrics_by_fold.png")
plot_feature_importances(rf_last_model.feature_importances_, feature_names,"Random Forest", f"Fold {len(splits)}","18_rf_feature_importances.png")

##Feed-Forward Neural Network

In [ ]:
input_dim = X.shape[1]                                    # Number of features for FFNN
nn_results = []
nn_fold_predictions = {}
all_epoch_losses = {}

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
    X_train, y_train, X_test, y_test, _ = scale_split(X, y, train_idx, test_idx)
    model, epoch_losses = _train_nn(X_train, y_train, input_dim)
    y_pred, y_prob = _predict(model, X_test)
    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics["split"] = split_num
    nn_results.append(metrics)
    nn_fold_predictions[split_num] = {"y_pred": y_pred, "y_prob": y_prob}
    all_epoch_losses[split_num] = epoch_losses

nn_dset2_df = pd.DataFrame(nn_results)
nn_dset2_df.to_csv(os.path.join(RESULTS, "nn_results.csv"), index=False)
print_summary_table(nn_dset2_df, "Feed-Forward Neural Network")

In [ ]:
# Plot training loss curves for each fold and produce evaluation plots for Dataset 2 NN.
fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.get_cmap("tab10")
for split_num in sorted(all_epoch_losses.keys()):
    losses = all_epoch_losses[split_num]
    ax.plot(range(1, len(losses) + 1), losses,
            label=f"Fold {split_num}", color=cmap((split_num - 1) % 10), linewidth=1.3)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Binary Cross-Entropy Loss", fontsize=12)
ax.set_title("Feed-Forward Neural Network - Training Loss Curves",
             fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=8); ax.grid(True, alpha=0.3)
fig.tight_layout()
save_fig(fig, "19_nn_training_loss.png")

plot_confusion_matrices(splits, X, y,_make_cached_predictor(nn_fold_predictions, "y_pred", len(splits)),"Feed-Forward Neural Network", "20_nn_confusion_matrices.png")
plot_roc_curves(splits, X, y,_make_cached_predictor(nn_fold_predictions, "y_prob", len(splits)),"Feed-Forward Neural Network", "21_nn_roc_curves.png")
plot_metrics_bars(nn_dset2_df, splits, "Feed-Forward Neural Network","22_nn_metrics_by_fold.png")
print("\nFeed-Forward Neural Network complete.")

In [ ]:
# Build a Dataset-2-wide summary table of mean metrics across folds for each model.
summary_frames = {"Logistic Regression": lr_dset2_df,"Random Forest": rf_dset2_df,"Feed-Forward NN": nn_dset2_df}
metric_cols = ["precision", "recall", "f1", "mcc", "roc_auc"]
summary = pd.DataFrame({
    name: frame[metric_cols].mean()
    for name, frame in summary_frames.items()
}).T

print("Mean metrics across all 4 folds:")
print(summary.round(4))
summary.to_csv(os.path.join(RESULTS, "summary_mean_metrics.csv"))

# Dataset 3
Classic credit-card fraud dataset — extremely imbalanced. Uses TIME-based expanding-window splits (instead of K-Fold) to better simulate real-world deployment, and SMOTE on the training portion only.

### Plan of Action
1. Load Data from kaggle
2. Clean and display information about it
3. Time-based expanding window splits and SMOTE(This is on training data only btw)
4. Logistic Regression - 4 Plots
5. Random Forest
6. FFNN

In [ ]:
RD = "sample_data/results/Dataset3"                       # Custom path used elsewhere; not the active RESULTS dir
os.makedirs(RD, exist_ok=True)

## Data Cleaning

In [ ]:
Dataset3["Class"] = Dataset3["Class"].astype(int)         # Force the label to integer
missing = Dataset3.isnull().sum()                         # Count missing per column
total_missing = missing.sum()
if total_missing>0:
  Dataset3 = Dataset3.dropna().reset_index(drop=True)     # Drop any rows with NaNs
duplicates = Dataset3.duplicated().sum()                  # Count duplicate rows
if duplicates>0:
  Dataset3 = Dataset3.drop_duplicates().reset_index(drop=True)  # Remove duplicates

# Data Analysis

## Class Distribution in Bar and Pie Chart

In [ ]:
# Visualize the severe class imbalance present in Dataset 3.
class_counts = Dataset3["Class"].value_counts().sort_index()
class_pct = Dataset3["Class"].value_counts(normalize=True).sort_index() * 100

colors = ["#056517","#de1a24"]
labels = ["Legitimate", "Fraudulent"]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(labels, class_counts.values, color=colors, edgecolor="black")
axes[0].set_title("Transaction Class Counts", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Count")
for i in range(len(labels)):
    count = class_counts.values[i]
    pct   = class_pct.values[i]
    y_position = count + count * 0.01                     # Offset annotation slightly above bar
    label_text = f"{count:,}\n({pct:.3f}%)"
    axes[0].text(i, y_position, label_text, ha="center", fontsize=10)

# Explode the fraud slice in the pie chart so it's visible despite tiny size.
axes[1].pie(class_counts.values,labels=labels,colors=colors,autopct="%1.3f%%",startangle=90,explode=(0, 0.1),)
axes[1].set_title("Class Proportion", fontsize=14, fontweight = "bold")
save_fig(fig, "Dataset_Class_Distribution")

# All Transactions
These two graphs split up all transactions over time and all fraudulent transactions over the same length of time.

In [ ]:
Dataset3["Hour"] = Dataset3["Time"]/3600                  # Convert 'Time' (seconds since first txn) to hours
total_hours = Dataset3["Hour"].max()                      # Max hour value (data spans ~48 hours)
fig, axes = plt.subplots(1, 2, figsize=(12, 8), sharex = True) # Sharex
axes[0].hist(Dataset3["Hour"], bins = 48, color = "#056517", edgecolor = "black")
axes[0].set_title(" All Transactions Over Time", fontsize = 14, fontweight = "bold")
axes[0].set_xlabel("Number of Transactions")

fraud_hours = Dataset3[Dataset3["Class"] == 1]["Hour"]    # Time-of-occurrence for fraud only
axes[1].hist(fraud_hours, bins = 48, color = "#de1a24", edgecolor = "black")
axes[1].set_title("Fraudulent Transactions Over Time", fontsize = 14, fontweight = "bold")
axes[1].set_ylabel("Number of Transactions")
axes[1].set_xlabel("Time")
save_fig(fig, "All_Transactions_Over_Time")

#Feature Correlation Chart
This cell is measuring which features have the strongest linear correlation with the fraud label and plots it out.

In [ ]:
v_cols = [f"V{i}" for i in range (1,29)]                  # Anonymous PCA features V1..V28
feature_cols = v_cols + ["Amount"]
corr_class = (Dataset3[feature_cols + ["Class"]].corr()["Class"].drop("Class").sort_values())  # Correlation w/ label
fig, ax = plt.subplots(figsize = (12,6))
bar_colors = ["#de1a24" if v<0 else "#056517" for v in corr_class.values]
ax.barh(corr_class.index, corr_class.values, color = bar_colors, edgecolor = "black")
ax.set_xlabel("Peasron Correlation with Class")
ax.set_title("Feature Correlation with Class", fontsize = 14, fontweight = "bold")
save_fig(fig, "Feature_Correlation_Chart")

#Preprocessing and Time Splits
Implements time-based expanding-window cross-validation: each fold trains on all transactions before time T and tests on the next 8-hour window. This mimics deploying a model on past data and predicting future fraud.

In [ ]:
Dataset3.drop(columns = ["Hour"], inplace=True, errors ="ignore")#inplace  # Drop helper col, ignore if missing
X = Dataset3.drop(columns = ["Class"])                    # Feature matrix (still includes Time)
y = Dataset3["Class"]                                     # Binary fraud label

def create_time_splits(X, test_window = 8, train_hours=16):
  # Build expanding-window time splits: train on everything before `test_start`,
  # test on the next `test_window` hours, then advance.
  hours = X["Time"]/3600                                  # Time in hours
  total_hours = hours.max()
  splits = []
  test_start = train_hours                                # Start testing after the initial 16h training window

  while test_start < total_hours:
    test_end = min(test_start +test_window, total_hours+1)
    train_mask = hours<test_start                         # All earlier transactions become training set
    test_mask = (hours>=test_start) & (hours<test_end)    # Transactions in the next window are test set
    train_idx = X[train_mask].index.tolist()              # Index labels (NOT positions) — important for .loc later
    test_idx = X[test_mask].index.tolist()
    if len(train_idx)>0 and len(test_idx) >0:
      splits.append((train_mask, test_mask))              # NOTE: appending boolean masks but downstream code passes indices

    test_start += test_window                             # Slide the test window forward
  return splits


def scale_resample(Xfeatures, y, train_idx, test_idx):
  # Per-fold preprocessing for Dataset 3: scale features and SMOTE-oversample the train fold.
  feature_names = Xfeatures.columns.tolist()

  X_trainr = Xfeatures.iloc[train_idx]                    # Position-based selection
  X_testr = Xfeatures.iloc[test_idx]
  y_trainr = y.iloc[train_idx]
  y_testr = y.iloc[test_idx]
  scaler = StandardScaler()

  XtrainScaled = pd.DataFrame(scaler.fit_transform(X_trainr), columns= feature_names, index=X_trainr.index)
  XtestScaled = pd.DataFrame(scaler.transform(X_testr), columns = feature_names, index = X_testr.index)

  smote = SMOTE(random_state=42)
  X_train_resampled, y_train_resampled = smote.fit_resample(XtrainScaled, y_trainr)

  return X_train_resampled, y_train_resampled, XtestScaled, y_testr

splits = create_time_splits(X)                            # Generate the time-based splits

Had to redo my functions because there were slightly different than what was required

These `j*` versions use `.loc[]` (label-based indexing) instead of `.iloc[]` because the time-based splits store boolean masks → DataFrame indices, not positional indices.

In [ ]:


def jconfusion_matrices(splits, Xfeatures, y, train_predict_fn, model_name, file):
    # Same as confusion_matrices() but uses .loc to support label-indexed splits.
    fig, axes = plt.subplots(1, len(splits), figsize=(5 * len(splits), 4))
    if len(splits) == 1:
        axes = [axes]

    for i, (train_idx, test_idx) in enumerate(splits):
        y_pred = train_predict_fn(train_idx, test_idx)
        y_test = y.loc[test_idx]                          # Label-based indexing
        cm = confusion_matrix(y_test, y_pred)

        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i],
                    xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"])
        axes[i].set_title(f"Split {i + 1}")
        axes[i].set_xlabel("Predicted")
        axes[i].set_ylabel("Actual")

    fig.suptitle(f"{model_name} - Confusion Matrices", fontweight="bold")
    fig.tight_layout()
    save_fig(fig, file)


def jplot_roc_curve(splits, Xfeatures, y, train_predict_proba_fn, model_name, file):
    # Label-indexed ROC curve plotter (Dataset 3 variant).
    fig, ax = plt.subplots(figsize=(8, 6))

    for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
        y_prob = train_predict_proba_fn(train_idx, test_idx)
        y_test = y.loc[test_idx]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc_val = roc_auc_score(y_test, y_prob)
        ax.plot(fpr, tpr, label=f"Split {split_num} (AUC = {auc_val:.3f})")

    ax.plot([0, 1], [0, 1], "k--", label="Random Classifier")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{model_name} ROC Curve")
    ax.legend(loc="lower right")
    fig.tight_layout()
    save_fig(fig, file)

def jscale_resample(X, y, train_idx, test_idx):
    # Label-indexed scale + SMOTE pipeline used by Dataset-3 model loops.
    feature_names = X.columns.tolist()
    X_train_raw = X.loc[train_idx]                        # Use .loc because train_idx may be a boolean mask
    X_test_raw = X.loc[test_idx]
    y_train = y.loc[train_idx]
    y_test = y.loc[test_idx]

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_raw))
    X_test_scaled = pd.DataFrame(scaler.transform(X_test_raw))

    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

    return X_train_resampled, y_train_resampled, X_test_scaled, y_test


# MODELS

## Logistic Regression

In [ ]:
Xfeatures = X.drop(columns = ["Time"])                    # Drop raw 'Time' so model doesn't latch onto it directly
feature_names = Xfeatures.columns.tolist()
lr_results = []
lm = None                                                 # Will hold last LR model for coefficient plot
for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
  X_train_resampled, y_train_resampled, XtestScaled, y_testr = jscale_resample(Xfeatures, y, train_idx, test_idx)
  model = _build_lr()
  model.fit(X_train_resampled, y_train_resampled)
  y_pred = model.predict(XtestScaled)
  y_prob = model.predict_proba(XtestScaled)[:,1]
  metrics = compute_metrics(y_testr, y_pred, y_prob)
  lr_results.append(metrics)
lm = model                                                # Cache last fold's model
lr_results = pd.DataFrame(lr_results)

### Visualizations

In [ ]:
def _lr_train_predict(train_idx, test_idx):
    # Re-trains LR on a fold and returns hard predictions.
    X_tr, y_tr, X_te, _, = jscale_resample(Xfeatures, y, train_idx, test_idx)
    model = _build_lr()
    model.fit(X_tr, y_tr)
    return model.predict(X_te)

def _lr_train_predict_proba(train_idx, test_idx):
    # Re-trains LR on a fold and returns fraud probabilities.
    X_tr, y_tr, X_te, _, = jscale_resample(Xfeatures, y, train_idx, test_idx)
    model = _build_lr()
    model.fit(X_tr, y_tr)
    return model.predict_proba(X_te)[:, 1]

jconfusion_matrices(splits, Xfeatures, y, _lr_train_predict, "Logistic Regression", "Logistic_Regression_Confusion_Matrix")
jplot_roc_curve(splits, Xfeatures, y, _lr_train_predict_proba,"Logistic Regression", "Logistic_Regression_ROC_Curve")
plot_metrics_bar(pd.DataFrame(lr_results), splits,"Logistic Regression", "Logistic_Regression_Metrics")
plot_feature_coefficients(lm.coef_[0], feature_names,"Logistic Regression", f"Split {len(splits)}", "Datadet3_LR_feature_coefficients.png")

# Random Forest

In [ ]:
rf_results = []
rf_last_model = None

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
  X_train_resampled, y_train_resampled, XtestScaled, y_testr = jscale_resample(Xfeatures, y, train_idx, test_idx)
  model = _build_rf()
  model.fit(X_train_resampled, y_train_resampled)
  y_pred = model.predict(XtestScaled)
  y_prob = model.predict_proba(XtestScaled)[:, 1]
  metrics = compute_metrics(y_testr, y_pred, y_prob)
  metrics["split"] = split_num
  rf_results.append(metrics)
  rf_last_model = model                                   # Save last model for feature importance plot

rf_df = pd.DataFrame(rf_results)

### Visualizations

In [ ]:
def _rf_train_predict(train_idx, test_idx):
  # Re-train RF on a fold and return predictions (used by jconfusion_matrices).
  X_tr, y_tr, X_te, _ = jscale_resample(Xfeatures, y, train_idx, test_idx)
  m = _build_rf()
  m.fit(X_tr, y_tr)
  return m.predict(X_te)

def _rf_train_predict_proba(train_idx, test_idx):
  # Re-train RF on a fold and return fraud probabilities (used by jplot_roc_curve).
  X_tr, y_tr, X_te, _ = jscale_resample(Xfeatures, y, train_idx, test_idx)
  m = _build_rf()
  m.fit(X_tr, y_tr)
  return m.predict_proba(X_te)[:, 1]

jconfusion_matrices(splits, Xfeatures, y, _rf_train_predict, "Random Forest", "RandomForestConfusionMatrices.png")
jplot_roc_curve(splits, Xfeatures, y, _rf_train_predict_proba, "Random Forest", "RandomForestROC.png")
plot_metrics_bar(rf_df, splits, "Random Forest", "RandomForestMetrics.png")

# Custom RF feature-importance plot (top 15) using the last fold's model.
importances = rf_last_model.feature_importances_
top_n = 15
sorted_idx = np.argsort(importances)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(range(top_n), importances[sorted_idx], color=["#de1a24" if i < 0 else "#056517" for i in importances[sorted_idx]], edgecolor="black")
ax.set_yticks(range(top_n))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel("Importance")
ax.set_title("Random Forest Feature Importances", fontsize=14, fontweight="bold")
fig.tight_layout()
save_fig(fig, "RandomForestFeatureImportances.png")

## FFNN Training, Evaluations and Visuals

In [ ]:
input_dim = Xfeatures.shape[1]                            # Number of input features for the FFNN
nn_results = []
split_predictions = {}                                    # Used by make_predictor() callbacks below
all_epoch_losses = {}

for split_num, (train_idx, test_idx) in enumerate(splits, start=1):
  X_train_resampled, y_train_resampled, XtestScaled, y_testr = jscale_resample(Xfeatures, y, train_idx, test_idx)

  model, epoch_losses = _train_nn(X_train_resampled, y_train_resampled, input_dim)  # Train one FFNN
  y_pred, y_prob = _predict(model, XtestScaled)           # Inference on this fold's test window
  metrics = compute_metrics(y_testr, y_pred, y_prob)
  metrics["split"] = split_num
  nn_results.append(metrics)
  split_predictions[split_num] = {"y_pred": y_pred, "y_prob": y_prob}
  all_epoch_losses[split_num] = epoch_losses

nn_results = pd.DataFrame(nn_results)

# Plot training loss curves for each time-based split.
fig, ax = plt.subplots(figsize=(10,5))
color_splits = ["#056517", "#de1a24", "#3b75e9", "#cad5ed"]
for split_num in sorted(all_epoch_losses.keys()):
  losses = all_epoch_losses[split_num]
  color = color_splits[(split_num-1)%len(color_splits)]   # Cycle through colors safely
  ax.plot(range(1, len(losses) +1), losses, label=f"Split {split_num}", color=color)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training Loss Over Epochs")
ax.legend()
save_fig(fig, "Training_Loss.png")
print(lr_results)
print(rf_df)
print(nn_results)
# Render evaluation plots — `make_predictor` returns cached predictions per fold.
plot_metrics_bar(nn_results,splits, "FFNN", "FFNN_Metrics")
jplot_roc_curve(splits, Xfeatures, y, make_predictor("y_prob"), "FFNN", "FFNN_ROC_Curve")
jconfusion_matrices(splits, Xfeatures, y, make_predictor("y_pred"), "FFNN", "FFNN_Confusion_Matrix")

In [ ]:
# Re-render the FFNN confusion matrices alone (rebuilds the cached predictor for a fresh call).
jconfusion_matrices(splits, Xfeatures, y, make_predictor("y_pred"), "FFNN", "FFNN_Confusion_Matrix")

# Analysis